<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_02_01_hpc_parallel_processing_r_base_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Data Processing and Analysis with {parallel}


As data sizes grow, traditional sequential data processing in R can become a bottleneck. Parallel computing allows you to leverage multiple CPU cores to speed up your data processing tasks. This tutorial will guide you through parallel data processing techniques in R with CPU cores, from basic concepts to practical applications.

The New York City Yellow Taxi Trip Data (`yellow_tripdata_2023.parquet`), provided by the NYC Taxi and Limousine Commission, is a rich dataset containing millions of trip records, including variables such as pickup and dropoff times, trip distances, fare amounts, and passenger counts. Analyzing such large datasets poses computational challenges, particularly for statistical tasks like descriptive statistics, correlation analysis, and regression modeling. The `{parallel}` package in R offers an efficient solution by distributing computations across multiple CPU cores, significantly reducing processing time for data-intensive operations.

This tutorial demonstrates how to leverage the `{parallel}` package to perform essential statistical analyses on the NYC taxi dataset. Specifically, we will compute descriptive statistics (e.g., mean, median, standard deviation) for key variables like trip duration, fare amount, and trip distance; calculate Pearson correlation coefficients to explore relationships between these variables; and fit both simple and multiple linear regression models to predict fare amounts based on trip duration, trip distance, and passenger count. By combining `{parallel}` with `{arrow}` for efficient Parquet file handling and `{dplyr}` for data manipulation, we ensure scalable and rapid analysis of large datasets. The tutorial uses `parLapply()` for cross-platform compatibility and `mclapply()` for macOS/Linux users, providing practical examples and performance comparisons to highlight the benefits of parallel processing.

##  Overview of the {parallel} Package

The `{parallel}` package is a core R package that provides functions for parallel computing, allowing you to utilize multiple CPU cores to speed up computations. It supports both multi-core processing on a single machine and distributed computing across multiple machines.

### Key Functions in {parallel}

- `mclapply()`: A parallel version of `lapply()` that works on Unix-like systems (Linux, macOS). It uses forking to create child processes. Not available on Windows.

- `parLapply()`: A parallel version of `lapply()` that works on all platforms, including Windows. It requires setting up a cluster using `makeCluster()`.

- `makeCluster()`: Creates a cluster of R processes that can run in parallel. You can specify the number of cores to use.

- `stopCluster()`: Stops the cluster and frees up resources.

- `detectCores()`: Detects the number of CPU cores available on the machine.

## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab's Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython

## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there. Adjust `dataFolder` in the data-setup cell to point at your Parquet file (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Check and Install Required R Packages

The cells below mirror the Quarto tutorial: define packages, install to Drive, verify, load.

### Define required packages

In [ ]:
%%R
packages <- c(
          'tidyverse',
          'data.table',
          'feather',
          'arrow',
          'parallel',
          'foreach',
          'doParallel',
          'parallelly',
          'nycflights13'
)

### Install missing packages (Google Drive library)

In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")

### Verify installation

In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))

### Load packages

In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages

### Check loaded packages

In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])

### Setting Up the {parallel} Package

First, let's check how many CPU cores are available:

In [ ]:
%%R
# Check number of CPU cores
num_cores <- detectCores()
cat("Available CPU cores:", num_cores, "\n")

# Recommended: use one less than total cores to avoid system slowdown
cores_to_use <- max(1, num_cores - 1)
cat("Recommended cores to use:", cores_to_use, "\n")

## Data Processing with {parallel}

This section demonstrates parallel data processing using the NYC Yellow Taxi Trip Data (`yellow_tripdata_2023.parquet`). We will compute the average trip distance per passenger count using both serial and parallel approaches with the `{parallel}` package. The dataset is large (~3 million rows), making it suitable for parallel processing to improve performance.

### Data

The dataset is in Parquet format, which can query directly without loading into memory using {arrow} package. For reproducibility:

- Download the January 2023 data from: [https://d37ci07v2hxiua.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet](https://d37ci07v2hxiua.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet) (about 47 MB, ~3 million rows).
- We saved it to a local folder, e.g., `/home/zia207/Dropbox/WebSites/R_Website/Quarto_Projects/R_Beginner/Data/yellow_tripdata_2023-01.parquet`.
- We'll also use the Taxi Zone Lookup CSV for joins: Download from [https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv](https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv).

The dataset includes columns like `VendorID`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `fare_amount`, `total_amount`, etc.

### Set `dataFolder` for Colab (edit path)

In [ ]:
%%R
# Colab: set folder containing yellow_tripdata_2023-01.parquet
dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
# Ensure trailing slash
dataFolder <- sub("/$", "", dataFolder)
dataFolder <- paste0(dataFolder, "/")

In [ ]:
%%R
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
# Load January 2023 data
taxi_data <- read_parquet(paste0(dataFolder, "yellow_tripdata_2023-01.parquet"))

Let's compute the average trip distance per `passenger_count` as an example. Since the dataset is large, we'll split it into chunks and process them in parallel using `parLapply()` (cross-platform) or `mclapply()` (macOS/Linux).

### Splitting the Data

To parallelize, we split the dataset into chunks based on the number of cores:

In [ ]:
%%R
# Number of chunks (equal to number of cores - 1)
n_chunks <- num_cores - 1

# Split data into chunks
data_splits <- split(taxi_data,
                     rep(1:n_chunks, each = ceiling(nrow(taxi_data) / n_chunks),
                         length.out = nrow(taxi_data)))

### Defining the Processing Function

Define a function to compute the average trip distance per passenger_count for each chunk:

In [ ]:
%%R
process_chunk <- function(chunk) {
  chunk %>%
    filter(!is.na(passenger_count), !is.na(trip_distance), passenger_count > 0) %>%
    group_by(passenger_count) %>%
    summarise(avg_distance = mean(trip_distance, na.rm = TRUE)) %>%
    collect()  # Ensure computation is performed
}

### Serial Processing (Baseline)

First, let's compute the result serially to establish a baseline

In [ ]:
%%R
system.time({
  serial_result <- lapply(data_splits, process_chunk)
  serial_result <- do.call(rbind, serial_result) %>%
    group_by(passenger_count) %>%
    summarise(avg_distance = mean(avg_distance, na.rm = TRUE))
})

print(serial_result)

### Parallel Processing with parLapply

Now, we'll parallelize the computation using `parLapply()`:

In [ ]:
%%R
# Create a cluster
cl <- makeCluster(num_cores - 1)

# Export required packages and functions to the cluster
clusterEvalQ(cl, {
  library(dplyr)
  library(arrow)
})
clusterExport(cl, c("process_chunk"))

# Parallel processing
system.time({
  parallel_result <- parLapply(cl, data_splits, process_chunk)
  parallel_result <- do.call(rbind, parallel_result) %>%
    group_by(passenger_count) %>%
    summarise(avg_distance = mean(avg_distance, na.rm = TRUE))
})

# Stop the cluster
stopCluster(cl)

print(parallel_result)

### Parallel Processing with mclapply (macOS/Linux)

If you're on macOS or Linux, you can use mclapply() for simpler parallelization:

In [ ]:
%%R
system.time({
  parallel_result_mc <- mclapply(data_splits, process_chunk, mc.cores = num_cores - 1)
  parallel_result_mc <- do.call(rbind, parallel_result_mc) %>%
    group_by(passenger_count) %>%
    summarise(avg_distance = mean(avg_distance, na.rm = TRUE))
})

print(parallel_result_mc)

## Basic Statistical Analysis

In this section of tutorial demonstrates how to perform basic statistical analyses on the NYC Yellow Taxi Trip Data (`yellow_tripdata_2023.parquet`) using R's {parallel} package to speed up computations. We'll cover descriptive statistics, correlation analysis, simple linear regression, and multiple linear regression to analyze relationships between trip duration, fare amount, trip distance, and passenger count. The {arrow} package will handle the Parquet file, and {dplyr} will assist with data manipulation. We'll parallelize computations to efficiently process the large dataset (~3 million rows).

We will conduct the following statistical analyses on the NYC Yellow Taxi data:

-   **Descriptive Statistics**: Compute mean, median, standard deviation, minimum, and maximum for trip_duration, fare_amount, and trip_distance.
-   **Correlation Analysis**: Calculate Pearson correlation coefficients among trip_duration, fare_amount, and trip_distance.
-   **Simple Linear Regression**: Estimate fare_amount based on trip_duration.
-   **Multiple Linear Regression**: Estimate fare_amount using trip_duration, trip_distance, and passenger_count.

We will employ the {parallel} package to parallelize computations across CPU cores, using `parLapply()` (a platform-independent function) and `mclapply()` (macOS/Linux).

### Read and Preprocess Data

In [ ]:
%%R
# Read and preprocess data
taxi_data <- read_parquet(paste0(dataFolder, "yellow_tripdata_2023.parquet")) %>%
  mutate(trip_duration = as.numeric(difftime(tpep_dropoff_datetime, tpep_pickup_datetime, units = "mins"))) %>%
  filter(
    !is.na(trip_duration), !is.na(fare_amount), !is.na(trip_distance), !is.na(passenger_count),
    trip_duration > 0, fare_amount > 0, trip_distance > 0, passenger_count > 0
  ) %>%
  select(trip_duration, fare_amount, trip_distance, passenger_count)

# Inspect the data
glimpse(taxi_data)

### Setting Up Parallel Processing

In [ ]:
%%R
# Detect number of CPU cores
num_cores <- detectCores()
print(num_cores)  # E.g., 8 cores

# Split data into chunks
n_chunks <- num_cores - 1
data_splits <- split(taxi_data, rep(1:n_chunks, each = ceiling(nrow(taxi_data) / n_chunks), length.out = nrow(taxi_data)))

### Descriptive Statistics

Compute mean, median, standard deviation, min, and max for trip_duration, fare_amount, and trip_distance.

#### Define the Descriptive Statistics Function

In [ ]:
%%R
process_descriptive <- function(chunk) {
  chunk %>%
    summarise(
      mean_duration = mean(trip_duration, na.rm = TRUE),
      median_duration = median(trip_duration, na.rm = TRUE),
      sd_duration = sd(trip_duration, na.rm = TRUE),
      min_duration = min(trip_duration, na.rm = TRUE),
      max_duration = max(trip_duration, na.rm = TRUE),
      mean_fare = mean(fare_amount, na.rm = TRUE),
      median_fare = median(fare_amount, na.rm = TRUE),
      sd_fare = sd(fare_amount, na.rm = TRUE),
      min_fare = min(fare_amount, na.rm = TRUE),
      max_fare = max(fare_amount, na.rm = TRUE),
      mean_distance = mean(trip_distance, na.rm = TRUE),
      median_distance = median(trip_distance, na.rm = TRUE),
      sd_distance = sd(trip_distance, na.rm = TRUE),
      min_distance = min(trip_distance, na.rm = TRUE),
      max_distance = max(trip_distance, na.rm = TRUE)
    ) %>%
    collect()
}

#### Serial Descriptive Statistics

In [ ]:
%%R
system.time({
  serial_desc <- lapply(data_splits, process_descriptive)
  serial_desc_summary <- do.call(rbind, serial_desc) %>%
    summarise(across(everything(), mean, na.rm = TRUE))
})

print(serial_desc_summary)

#### Parallel Descriptive Statistics with `parLapply`

In [ ]:
%%R
# Create a cluster
cl <- makeCluster(num_cores - 1)

# Export required packages and functions
clusterEvalQ(cl, {
  library(dplyr)
  library(arrow)
})
clusterExport(cl, c("process_descriptive"))

# Parallel descriptive statistics
system.time({
  parallel_desc <- parLapply(cl, data_splits, process_descriptive)
  parallel_desc_summary <- do.call(rbind, parallel_desc) %>%
    summarise(across(everything(), mean, na.rm = TRUE))
})

# Stop the cluster
stopCluster(cl)

print(parallel_desc_summary)

### Correlation Analysis

Compute the Pearson correlation matrix for `trip_duration`, `fare_amount`, and `trip_distance`.

#### Define the Correlation Function

In [ ]:
%%R
process_correlation <- function(chunk) {
  cor_matrix <- cor(chunk[, c("trip_duration", "fare_amount", "trip_distance")], method = "pearson", use = "complete.obs")
  as.vector(cor_matrix)
}

#### Serial Correlation

In [ ]:
%%R
system.time({
  serial_cor <- lapply(data_splits, process_correlation)
  serial_cor_mean <- colMeans(do.call(rbind, serial_cor), na.rm = TRUE)
  cor_matrix <- matrix(serial_cor_mean, nrow = 3, dimnames = list(
    c("trip_duration", "fare_amount", "trip_distance"),
    c("trip_duration", "fare_amount", "trip_distance")
  ))
})

print(round(cor_matrix, 3))

#### Parallel Correlation with `parLapply`

In [ ]:
%%R
cl <- makeCluster(num_cores - 1)
clusterEvalQ(cl, {
  library(dplyr)
  library(arrow)
})
clusterExport(cl, c("process_correlation"))

system.time({
  parallel_cor <- parLapply(cl, data_splits, process_correlation)
  parallel_cor_mean <- colMeans(do.call(rbind, parallel_cor), na.rm = TRUE)
  parallel_cor_matrix <- matrix(parallel_cor_mean, nrow = 3, dimnames = list(
    c("trip_duration", "fare_amount", "trip_distance"),
    c("trip_duration", "fare_amount", "trip_distance")
  ))
})

stopCluster(cl)

print(round(parallel_cor_matrix, 3))

#### Parallel Correlation with mclapply (macOS/Linux)

In [ ]:
%%R
system.time({
  parallel_cor_mc <- mclapply(data_splits, process_correlation, mc.cores = num_cores - 1)
  parallel_cor_mc_mean <- colMeans(do.call(rbind, parallel_cor_mc), na.rm = TRUE)
  parallel_cor_mc_matrix <- matrix(parallel_cor_mc_mean, nrow = 3, dimnames = list(
    c("trip_duration", "fare_amount", "trip_distance"),
    c("trip_duration", "fare_amount", "trip_distance")
  ))
})

print(round(parallel_cor_mc_matrix, 3))

### Simple Linear Regression

Model `fare_amount` as a function of `trip_duration`

#### Define the Simple Regression Function

In [ ]:
%%R
process_simple_regression <- function(chunk) {
  model <- lm(fare_amount ~ trip_duration, data = chunk)
  coef <- summary(model)$coefficients
  list(
    intercept = coef[1, 1],
    slope = coef[2, 1],
    r_squared = summary(model)$r.squared
  )
}

#### Serial Simple Regression

In [ ]:
%%R
system.time({
  serial_reg <- lapply(data_splits, process_simple_regression)
  serial_reg_summary <- do.call(rbind, lapply(serial_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope = mean(slope, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(serial_reg_summary)

#### Parallel Simple Regression with `parLapply`

In [ ]:
%%R
cl <- makeCluster(num_cores - 1)
clusterEvalQ(cl, {
  library(dplyr)
  library(arrow)
})
clusterExport(cl, c("process_simple_regression"))

system.time({
  parallel_reg <- parLapply(cl, data_splits, process_simple_regression)
  parallel_reg_summary <- do.call(rbind, lapply(parallel_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope = mean(slope, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

stopCluster(cl)

print(parallel_reg_summary)

#### Parallel Simple Regression with mclapply (macOS/Linux)

In [ ]:
%%R
system.time({
  parallel_reg_mc <- mclapply(data_splits, process_simple_regression, mc.cores = num_cores - 1)
  parallel_reg_mc_summary <- do.call(rbind, lapply(parallel_reg_mc, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope = mean(slope, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(parallel_reg_mc_summary)

### Multiple Linear Regression

Model `fare_amount` as a function of `trip_duration`, `trip_distance`, and `passenger_count`

#### Define the Multiple Regression Function

In [ ]:
%%R
process_multiple_regression <- function(chunk) {
  model <- lm(fare_amount ~ trip_duration + trip_distance + passenger_count, data = chunk)
  coef <- summary(model)$coefficients
  list(
    intercept = coef[1, 1],
    slope_duration = coef[2, 1],
    slope_distance = coef[3, 1],
    slope_passenger = coef[4, 1],
    r_squared = summary(model)$r.squared
  )
}

#### Serial Multiple Regression

In [ ]:
%%R
system.time({
  serial_multi_reg <- lapply(data_splits, process_multiple_regression)
  serial_multi_reg_summary <- do.call(rbind, lapply(serial_multi_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope_duration = mean(slope_duration, na.rm = TRUE),
      avg_slope_distance = mean(slope_distance, na.rm = TRUE),
      avg_slope_passenger = mean(slope_passenger, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(serial_multi_reg_summary)

#### Parallel Multiple Regression with `parLapply`

In [ ]:
%%R
cl <- makeCluster(num_cores - 1)
clusterEvalQ(cl, {
  library(dplyr)
  library(arrow)
})
clusterExport(cl, c("process_multiple_regression"))

system.time({
  parallel_multi_reg <- parLapply(cl, data_splits, process_multiple_regression)
  parallel_multi_reg_summary <- do.call(rbind, lapply(parallel_multi_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope_duration = mean(slope_duration, na.rm = TRUE),
      avg_slope_distance = mean(slope_distance, na.rm = TRUE),
      avg_slope_passenger = mean(slope_passenger, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

stopCluster(cl)

print(parallel_multi_reg_summary)

#### Parallel Multiple Regression with mclapply (macOS/Linux)

In [ ]:
%%R
system.time({
  parallel_multi_reg_mc <- mclapply(data_splits, process_multiple_regression, mc.cores = num_cores - 1)
  parallel_multi_reg_mc_summary <- do.call(rbind, lapply(parallel_multi_reg_mc, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope_duration = mean(slope_duration, na.rm = TRUE),
      avg_slope_distance = mean(slope_distance, na.rm = TRUE),
      avg_slope_passenger = mean(slope_passenger, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(parallel_multi_reg_mc_summary)

## Key points to Consider When Using Parallel Processing in R

### Rule of thumb: only parallelize when tasks are substantial

In [ ]:
%%R
# Rule of thumb: only parallelize when tasks are substantial
should_parallelize <- function(task_time, setup_time = 0.1, overhead_factor = 2) {
  # Task should be at least 10x the setup time
  task_time > (setup_time * overhead_factor * 10)
}

# Example usage
task_time <- 0.5  # seconds
if(should_parallelize(task_time)) {
  cat("Consider parallel processing\n")
} else {
  cat("Sequential processing is sufficient\n")
}

### Memory Management

In [ ]:
%%R
# Monitor memory usage
get_memory_usage <- function() {
  if(Sys.info()["sysname"] == "Windows") {
    # Windows-specific memory check
    mem <- system("tasklist /FI \"IMAGENAME eq Rterm.exe\" /FO CSV /NH",
                  intern = TRUE)
    # Parse memory from output
    return("Memory monitoring not implemented for Windows")
  } else {
    # Unix/Linux/Mac
    mem_kb <- system("ps -o rss= -p $(pgrep R)", intern = TRUE)
    return(as.numeric(mem_kb) / 1024)  # Convert to MB
  }
}

# Use garbage collection
perform_gc <- function() {
  gc(verbose = FALSE)
  cat("Garbage collection performed\n")
}

### Error Handling and Debugging

In [ ]:
%%R
# Robust parallel processing with error handling
safe_parallel_process <- function(data_list, processor_func, n_cores = NULL) {
  if(is.null(n_cores)) {
    n_cores <- max(1, detectCores() - 1)
  }

  # Set up cluster
  cl <- makeCluster(n_cores)
  clusterExport(cl, "processor_func")

  # Process with error handling
  tryCatch({
    results <- parLapply(cl, data_list, function(chunk) {
      tryCatch({
        processor_func(chunk)
      }, error = function(e) {
        list(error = TRUE, message = e$message, traceback = traceback())
      })
    })
  }, error = function(e) {
    cat("Parallel processing failed:", e$message, "\n")
    results <- NULL
  })

  # Clean up
  stopCluster(cl)

  return(results)
}

## Best Practices

-   `Filter early`: Remove NA values and invalid data (e.g., passenger_count <= 0) before splitting to save memory.

-   `Optimize chunks`: Make sure chunks are balanced (similar row counts) to improve core utilization.

-   `Memory management`: Large datasets like NYC taxi data can use a lot of memory. Use {arrow}'s memory-efficient Parquet reading to reduce RAM usage.

-   `Cluster cleanup`: Always call stopCluster(cl) after using parLapply() to free resources.

-   `Test serially`: Debug with lapply() before switching to parallel to ensure correctness.

## Limitations

-   `Windows`: Use parLapply() because mclapply() is not supported.

-   `Overhead`: Parallelization has setup costs (e.g., cluster creation, data splitting). It is only beneficial for computationally intensive tasks.

-   `I/O bottlenecks`: Reading the Parquet file is not parallelized here; consider parallel I/O with tools like {duckdb} for very large datasets.

## parallelly: Enhancing the {parallel} Package

The `parallelly` package in R is an enhancement of the base R `parallel` package, designed to improve and extend parallel computing capabilities. Part of the Futureverse ecosystem, `parallelly` provides additional tools for managing parallel workers, including querying, killing, and cloning nodes, with support for both local and remote machines. It is backward compatible with the `parallel` package, ensuring seamless integration, and adds features like automatic detection of CPU cores, simplified remote cluster setup, and automatic worker cleanup. Key functions include `availableCores()`, `makeClusterPSOCK()`, `isNodeAlive()`, `killNode()`, and `cloneNode()`, which offer more robust control over parallel processes compared to the base `parallel` package.

### Differences from the `parallel` Package

1. **Enhanced CPU Core Detection**: `parallelly`'s `availableCores()` considers system settings, cgroups, Linux containers, and job scheduler configurations, falling back to `parallel::detectCores()` if needed, unlike the simpler `detectCores()` in `parallel`.

2. **Improved Remote Cluster Setup**: `makeClusterPSOCK()` in `parallelly` simplifies remote worker setup without requiring manual IP address or firewall configuration, unlike `parallel::makePSOCKcluster()`.

3. **Advanced Worker Management**: `parallelly` introduces `isNodeAlive()` and `killNode()` for checking and terminating workers (local and remote) and `cloneNode()` for restarting failed nodes, features absent in `parallel`.

4. **Automatic Cleanup**: `parallelly::autoStopCluster()` ensures clusters shut down automatically when garbage-collected, reducing stray processes, unlike `parallel`.

5. **Futureverse Integration**: While `parallelly` supports the `future` package, it can be used independently, unlike `parallel`, which is more tightly coupled to base R.

Below is an example of using `parallelly` to process the NYC Taxi `yellow_tripdata_2023-01.parquet` dataset, which contains taxi trip data. The example reads the Parquet file, processes trip distances in parallel to calculate total distance per passenger count, and compares it to sequential processing.

In [ ]:
%%R
# Function to compute total trip distance per passenger count
calc_distance <- function(df_chunk) {
  aggregate(trip_distance ~ passenger_count, data = df_chunk, sum)
}

# Sequential processing
system.time({
  result_seq <- calc_distance(taxi_data)
})

In [ ]:
%%R
# Inspect the data
print("Dataset summary:")
summary(taxi_data[, c("passenger_count", "trip_distance")])

# Filter out invalid rows (e.g., NA or negative passenger_count/trip_distance)
taxi_data <- taxi_data[!is.na(taxi_data$passenger_count) &
                       !is.na(taxi_data$trip_distance) &
                       taxi_data$passenger_count > 0 &
                       taxi_data$trip_distance >= 0, ]

# Check if data is non-empty after filtering
if (nrow(taxi_data) == 0) {
  stop("No valid data after filtering. Check passenger_count and trip_distance columns.")
}

# Function to compute total trip distance per passenger count
calc_distance <- function(df_chunk) {
  # Return NULL if the chunk is empty or has no valid rows
  if (nrow(df_chunk) == 0 || all(is.na(df_chunk$passenger_count)) || all(is.na(df_chunk$trip_distance))) {
    return(NULL)
  }
  # Aggregate only if valid data exists
  aggregate(trip_distance ~ passenger_count, data = df_chunk, sum, na.rm = TRUE)
}

In [ ]:
%%R
# Sequential processing
system.time({
  result_seq <- calc_distance(taxi_data)
})

In [ ]:
%%R
# Parallel processing with parallelly
system.time({
  # Create a cluster with parallelly, limiting to available connections
  max_cores <- min(availableCores() - 1, 45) # Use at most 45 cores
  cl <- makeClusterPSOCK(max_cores)
  on.exit(parallelly::autoStopCluster(cl)) # Ensure cluster cleanup

  # Split data into chunks, ensuring non-empty chunks
  n_cores <- length(cl)
  data_chunks <- split(taxi_data, cut(seq_len(nrow(taxi_data)), n_cores, labels = FALSE))

  # Export function and required objects to cluster
  clusterExport(cl, c("calc_distance"))

  # Parallel computation
  results <- parLapply(cl, data_chunks, calc_distance)

  # Filter out NULL results and combine
  results <- results[!sapply(results, is.null)] # Remove empty results
  if (length(results) == 0) {
    stop("No valid results from parallel computation. Check data chunks.")
  }
  result_par <- do.call(rbind, results)
  result_par <- aggregate(trip_distance ~ passenger_count, data = result_par, sum, na.rm = TRUE)
})

# Print results
print("Sequential Result:")
print(result_seq)
print("Parallel Result:")
print(result_par)

## Summary and Conclusion

Parallel data processing is a critical technique for managing large-scale data and computationally intensive tasks. By distributing workloads across multiple processing units, it achieves significant speedups, scalability, and efficiency compared to sequential methods. It relies on effective data partitioning, task coordination, and robust frameworks like Spark or Hadoop. As data volumes grow and computational demands increase, parallel processing remains essential for industries and research fields requiring fast, scalable solutions.

The {parallel} package enables efficient computation of descriptive statistics, correlation, and regression analyses on large datasets like the NYC Yellow Taxi data. By splitting the data and using parLapply() or mclapply(), we analyzed trip_duration, fare_amount, trip_distance, and passenger_count with significant speedups. Combining {parallel} with {arrow} ensures scalability. For further analysis, consider grouping by other variables or exploring nonlinear models.

## References

1.  **R Core Team (2023). "parallel: Support for Parallel Computation in R."**
    -   **Description**: Official documentation for the `{parallel}` package, included in base R. It provides detailed explanations of functions like `mclapply()` and `parLapply()`, with examples for parallelizing computations.
    -   **Source**: Available in R via `help(package = "parallel")` or online at https://cran.r-project.org/web/packages/parallel/parallel.pdf
    -   **Relevance**: Primary reference for understanding the `{parallel}` package's functionality and implementation.
2.  **McCallum, E., & Weston, S. (2011). "Parallel R." O'Reilly Media.**
    -   **Description**: A practical guide to parallel computing in R, covering the `{parallel}` package and its predecessors (`snow`, `multicore`). It includes examples of parallelizing data analysis tasks.
    -   **Source**: Available as a book or through O'Reilly's online platform: https://www.oreilly.com/library/view/parallel-r/9781449309909/
    -   **Relevance**: Offers practical insights into applying `{parallel}` for data analysis, including large dataset processing.
3.  **Mayer, M. (2017). "Parallel Computing in R with the parallel Package."**
    -   **Description**: A tutorial blog post that explains how to use the `{parallel}` package for tasks like data processing and statistical analysis, with code examples for `parLapply()` and `mclapply()`.
    -   **Source**: Available at https://www.r-bloggers.com/2017/03/parallel-comte-in-r/
    -   **Relevance**: Provides beginner-friendly examples of parallelizing statistical computations, relevant to correlation and regression.
4.  **NYC Taxi and Limousine Commission (2023). "TLC Trip Record Data."**
    -   **Description**: Official source for the NYC Yellow Taxi Trip Data, including the `yellow_tripdata_2023-01.parquet` file. It includes metadata and documentation for the dataset used in the analysis.
    -   **Source**: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
    -   **Relevance**: Essential for understanding the structure and variables of the taxi dataset used in the statistical analyses.
5.  **Wickham, H., & Grolemund, G. (2016). "R for Data Science." O'Reilly Media.**
    -   **Description**: A comprehensive guide to data analysis in R, including sections on data manipulation with `{dplyr}` and handling large datasets. While not specific to `{parallel}`, it provides context for combining `{dplyr}` with parallel processing.
    -   **Source**: Available online at https://r4ds.had.co.nz/ or as a book.
    -   **Relevance**: Useful for understanding `{dplyr}` operations (e.g., filtering, grouping) used in conjunction with `{parallel}` for statistical analysis.